
# Random Forest - Heart Attack Dataset

## Ciência de Dados

Projeto de classificação supervisionada utilizando o algoritmo Random Forest para prever risco de ataque cardíaco.



# Importação das Bibliotecas

Nesta etapa são importadas as bibliotecas necessárias para:

- manipulação da base de dados
- treinamento do modelo
- avaliação das métricas
- geração de gráficos
- ajuste de hiperparâmetros


In [ ]:

import random
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)



# Geração da Seed Aleatória

A seed é utilizada para garantir reprodutibilidade parcial dos resultados.

O valor utilizado será exibido durante a execução.


In [ ]:

SEMENTE = random.randint(1, 100000)

print("\n==============================")
print("SEMENTE UTILIZADA")
print("==============================\n")

print(SEMENTE)



# Carregamento do Dataset

O dataset deve estar na mesma pasta do notebook com o nome:

```text
Heart_Attack_Data_Set.csv
```


In [ ]:

df = pd.read_csv("Heart_Attack_Data_Set.csv")



# Visualização Inicial da Base

Exibição das primeiras linhas e informações gerais do dataset.


In [ ]:

print("\n==============================")
print("PRIMEIRAS LINHAS")
print("==============================\n")

print(df.head())

print("\n==============================")
print("INFORMAÇÕES DA BASE")
print("==============================\n")

print(df.info())



# Separação das Variáveis

- `X` contém as variáveis explicativas
- `y` contém a variável alvo (`target`)


In [ ]:

X = df.drop(columns=["target"])

y = df["target"]



# Divisão entre Treino e Teste

A base será dividida em:

- 80% para treino
- 20% para teste

O parâmetro `stratify=y` mantém o balanceamento das classes.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEMENTE,
    stratify=y
)



# Definição dos Hiperparâmetros

O GridSearchCV irá testar diferentes combinações de hiperparâmetros para encontrar a melhor configuração do Random Forest.


In [ ]:

parametros = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}



# Criação do Modelo

Inicialização do algoritmo Random Forest.


In [ ]:

modelo = RandomForestClassifier(
    random_state=SEMENTE
)



# Ajuste Automático de Hiperparâmetros

O GridSearchCV realiza validação cruzada para encontrar a melhor configuração do modelo.


In [ ]:

grid_search = GridSearchCV(
    estimator=modelo,
    param_grid=parametros,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

melhor_modelo = grid_search.best_estimator_



# Predições

O modelo treinado será utilizado para prever os dados de teste.


In [ ]:

y_pred = melhor_modelo.predict(X_test)



# Cálculo das Métricas

As métricas utilizadas serão:

- acurácia
- precisão
- revocação
- F1-score
- matriz de confusão


In [ ]:

acuracia = accuracy_score(y_test, y_pred)

precisao = precision_score(y_test, y_pred)

revocacao = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

matriz = confusion_matrix(y_test, y_pred)



# Resultados do Modelo

Exibição dos melhores hiperparâmetros encontrados e das métricas finais.


In [ ]:

print("\n==============================")
print("MELHORES HIPERPARÂMETROS")
print("==============================\n")

print(grid_search.best_params_)

print("\n==============================")
print("RESULTADOS RANDOM FOREST")
print("==============================\n")

print(f"Acurácia: {acuracia:.4f}")

print(f"Precisão: {precisao:.4f}")

print(f"Revocação: {revocacao:.4f}")

print(f"F1-Score: {f1:.4f}")

print("\n==============================")
print("MATRIZ DE CONFUSÃO")
print("==============================\n")

print(matriz)

print("\n==============================")
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("==============================\n")

print(classification_report(y_test, y_pred))



# Importância das Variáveis

O Random Forest permite identificar quais variáveis tiveram maior influência nas previsões.


In [ ]:

importancias = pd.DataFrame({
    "Variável": X.columns,
    "Importância": melhor_modelo.feature_importances_
})

importancias = importancias.sort_values(
    by="Importância",
    ascending=False
)

print("\n==============================")
print("IMPORTÂNCIA DAS VARIÁVEIS")
print("==============================\n")

print(importancias)



# Gráfico da Matriz de Confusão

Visualização gráfica dos acertos e erros do modelo.


In [ ]:

plt.figure(figsize=(8, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Sem Ataque", "Com Ataque"]
)

disp.plot(cmap="Blues")

plt.title("Matriz de Confusão")

plt.show()



# Gráfico das Variáveis Mais Importantes

Exibição das 10 variáveis com maior importância no modelo.


In [ ]:

top10 = importancias.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top10["Variável"],
    top10["Importância"]
)

plt.xlabel("Importância")

plt.ylabel("Variável")

plt.title("Top 10 Variáveis Mais Importantes")

plt.gca().invert_yaxis()

plt.show()
